# Iris Classifier using Vertex AI

### Inference

In [18]:
from google.cloud import storage
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import joblib
from sklearn import metrics

In [19]:
PROJECT_ID     = "sheetgpt-385916"
LOCATION       = "us-central1"
BUCKET_URI     = "gs://mlops-week1"
BUCKET_NAME    = "mlops-week1"
DATASET_LOCAL_PATH = "./data/data.csv"
LOCAL_MODEL_FILEPATH = "inference_artifacts/model.joblib"

In [20]:
client = storage.Client()

In [21]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

In [22]:
def write_to_gcs(bucket_name, log_message, gcs_path):
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(gcs_path)
    blob.upload_from_string(log_message, content_type="text/plain")
    print(f"Log written to gs://{bucket_name}/{gcs_path}")

In [23]:
def fetch_latest_from_gcs(bucket_name, parent_dir, file_name, destination_file):
    bucket = client.bucket(bucket_name)
    blobs = list(bucket.list_blobs(prefix=parent_dir))
    candidate_paths = [b.name for b in blobs if b.name.endswith(file_name)]

    if not candidate_paths:
        raise FileNotFoundError(f"No {file_name} found under gs://{BUCKET_NAME}/{parent_dir}")

    candidate_paths.sort()
    latest_blob_name = candidate_paths[-1]

    blob = bucket.blob(latest_blob_name)
    blob.download_to_filename(destination_file)

    print(f"Downloaded latest {file_name} from gs://{bucket_name}/{latest_blob_name} -> {destination_file}")
    
    return f'{latest_blob_name}'


In [24]:
data = pd.read_csv(DATASET_LOCAL_PATH)
data.head(5)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.8,4.0,1.2,0.2,setosa
1,5.7,4.4,1.5,0.4,setosa
2,5.4,3.9,1.3,0.4,setosa
3,5.1,3.5,1.4,0.3,setosa
4,5.7,3.8,1.7,0.3,setosa


In [25]:
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_eval = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_eval = test.species

##### Inference

In [30]:
path = fetch_latest_from_gcs(BUCKET_NAME, "v1", "model.joblib", LOCAL_MODEL_FILEPATH)

Downloaded latest model.joblib from gs://mlops-week1/v1/2025-10-04-210713/model.joblib -> inference_artifacts/model.joblib


In [31]:
model = joblib.load(LOCAL_MODEL_FILEPATH)

In [32]:
prediction=model.predict(X_eval)
log_message = f'The accuracy of the Model is {metrics.accuracy_score(prediction, y_eval):.3f}'
print(log_message)

The accuracy of the Model is 0.951


In [33]:
write_to_gcs(BUCKET_NAME, log_message, f'v1/{path.split("/")[1]}/metrics.log')
print(f'{log_message} successfully written to gs://{BUCKET_NAME}/v1/{path.split("/")[1]}/metrics.log')

Log written to gs://mlops-week1/v1/2025-10-04-210713/metrics.log
The accuracy of the Model is 0.951 successfully written to gs://mlops-week1/v1/2025-10-04-210713/metrics.log
